# Module 02: Pandas for Machine Learning
## Notebook 02: Indexing, Filtering, and Safe Assignment

Precise row and column selection is the core of exploratory data analysis and feature preprocessing. Slicing rows incorrectly or assigning values to views instead of copies causes subtle bugs and `SettingWithCopyWarning` errors in machine learning pipelines.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Distinguish between label-based (`.loc`) and integer position-based (`.iloc`) accessors.
2. Filter data with compound boolean masks (`&`, `|`, `~`).
3. Query datasets elegantly using `.isin()`, `.between()`, and `.query()`.
4. Understand and resolve the notorious `SettingWithCopyWarning`.
5. Safely create, modify, and assign engineered feature columns.
6. **Advanced:** Build modular method-chaining workflows with `.pipe()`, `.assign()`, dynamic querying, and multi-branch conditional vectorization (`np.select`).

In [1]:
import pandas as pd
import numpy as np

# Create a sample customer churn dataset
np.random.seed(42)
n_samples = 8

data = {
    'Account_ID': [f"ACC-{100+i}" for i in range(n_samples)],
    'Age': [24, 52, 36, 41, 67, 29, 38, 45],
    'Monthly_Spend': [45.2, 120.5, 89.0, 34.0, 150.2, 65.4, 112.0, 78.5],
    'Contract': ['Month-to-month', 'Two year', 'One year', 'Month-to-month', 'Two year', 'Month-to-month', 'One year', 'Month-to-month'],
    'Support_Tickets': [3, 0, 1, 4, 0, 2, 1, 5],
    'Churn': ['No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes']
}

df = pd.DataFrame(data)
df.set_index('Account_ID', inplace=True)
print("Initial Dataset:")
print(df)

Initial Dataset:
            Age  Monthly_Spend        Contract  Support_Tickets Churn
Account_ID                                                           
ACC-100      24           45.2  Month-to-month                3    No
ACC-101      52          120.5        Two year                0    No
ACC-102      36           89.0        One year                1    No
ACC-103      41           34.0  Month-to-month                4   Yes
ACC-104      67          150.2        Two year                0    No
ACC-105      29           65.4  Month-to-month                2    No
ACC-106      38          112.0        One year                1    No
ACC-107      45           78.5  Month-to-month                5   Yes


---
### 1. Label-Based (`.loc`) vs. Position-Based (`.iloc`) Selection

The golden rule of Pandas indexing:
- `.loc[row_labels, column_labels]`: Selects by **label** (inclusive of both start and end bounds).
- `.iloc[row_positions, column_positions]`: Selects by **0-indexed integer position** (exclusive of stop bound, identical to standard Python slicing).

In [2]:
# .loc: Select by label names
print("=== .loc: Single row by label ===")
print(df.loc['ACC-102'])

print("\n=== .loc: Multiple rows and specific columns ===")
print(df.loc[['ACC-100', 'ACC-104'], ['Monthly_Spend', 'Contract']])

# .iloc: Select by integer coordinates
print("\n=== .iloc: First 3 rows and first 2 columns ===")
print(df.iloc[0:3, 0:2])

=== .loc: Single row by label ===
Age                      36
Monthly_Spend          89.0
Contract           One year
Support_Tickets           1
Churn                    No
Name: ACC-102, dtype: object

=== .loc: Multiple rows and specific columns ===
            Monthly_Spend        Contract
Account_ID                               
ACC-100              45.2  Month-to-month
ACC-104             150.2        Two year

=== .iloc: First 3 rows and first 2 columns ===
            Age  Monthly_Spend
Account_ID                    
ACC-100      24           45.2
ACC-101      52          120.5
ACC-102      36           89.0


---
### 2. Boolean Filtering: Single & Compound Conditions

In Pandas, bitwise operators are mandatory for combining conditional Series:
- `&` represents AND
- `|` represents OR
- `~` represents NOT
- Every individual condition **must** be enclosed in parentheses `()` due to Python operator precedence.

In [3]:
# High spenders (> $80) who also have month-to-month contracts
high_spend_m2m = df[(df['Monthly_Spend'] > 80.0) & (df['Contract'] == 'Month-to-month')]
print("High-Spend Month-to-Month Customers:")
print(high_spend_m2m[['Monthly_Spend', 'Contract', 'Support_Tickets']])

# Customers either older than 60 OR having more than 3 support tickets
vulnerable_customers = df[(df['Age'] > 60) | (df['Support_Tickets'] > 3)]
print("\nHigh-Risk Customers (Age > 60 OR Tickets > 3):")
print(vulnerable_customers[['Age', 'Support_Tickets', 'Churn']])

High-Spend Month-to-Month Customers:
Empty DataFrame
Columns: [Monthly_Spend, Contract, Support_Tickets]
Index: []

High-Risk Customers (Age > 60 OR Tickets > 3):
            Age  Support_Tickets Churn
Account_ID                            
ACC-103      41                4   Yes
ACC-104      67                0    No
ACC-107      45                5   Yes


---
### 3. High-Level Filtering: `.isin()`, `.between()`, and `.query()`

Pandas provides declarative filtering helpers that make feature engineering cleaner:
- `.isin(values)`: Checks membership in a collection.
- `.between(left, right)`: Checks if values fall within numeric range (inclusive).
- `.query('expression')`: Evaluates a string boolean expression in DataFrame scope.

In [4]:
# 1. .isin()
long_contracts = df[df['Contract'].isin(['One year', 'Two year'])]
print("Committed Contract Customers:\n", long_contracts[['Contract', 'Monthly_Spend']])

# 2. .between()
mid_age_customers = df[df['Age'].between(30, 50)]
print("\nCustomers aged between 30 and 50:\n", mid_age_customers[['Age', 'Monthly_Spend']])

# 3. .query() with string expression
query_result = df.query("Monthly_Spend > 75 and Support_Tickets >= 1 and Churn == 'No'")
print("\nFiltered via df.query():\n", query_result[['Monthly_Spend', 'Support_Tickets', 'Churn']])

Committed Contract Customers:
             Contract  Monthly_Spend
Account_ID                         
ACC-101     Two year          120.5
ACC-102     One year           89.0
ACC-104     Two year          150.2
ACC-106     One year          112.0

Customers aged between 30 and 50:
             Age  Monthly_Spend
Account_ID                    
ACC-102      36           89.0
ACC-103      41           34.0
ACC-106      38          112.0
ACC-107      45           78.5

Filtered via df.query():
             Monthly_Spend  Support_Tickets Churn
Account_ID                                      
ACC-102              89.0                1    No
ACC-106             112.0                1    No


---
### 4. Avoiding the `SettingWithCopyWarning`

When you write `df[condition]['col'] = value`, Pandas executes chained indexing:
1. `df[condition]` returns a temporary slice (which may be a view or a copy).
2. `['col'] = value` assigns to that temporary slice.

This produces a `SettingWithCopyWarning` and frequently fails to update the original DataFrame!

**The Proper Fix:** Always use `.loc[condition, 'col'] = value` or explicitly `.copy()`.

In [5]:
# SAFE PATTERN 1: Using .loc for condition-based assignment
df.loc[df['Support_Tickets'] >= 3, 'High_Support_Flag'] = 1
df.loc[df['Support_Tickets'] < 3, 'High_Support_Flag'] = 0

print("DataFrame after safe .loc assignment:")
print(df[['Support_Tickets', 'High_Support_Flag']])

# SAFE PATTERN 2: Working on a detached subset using .copy()
subset_df = df[df['Contract'] == 'Two year'].copy()
subset_df['Discount_Applied'] = subset_df['Monthly_Spend'] * 0.90
print("\nIsolated subset DataFrame:\n", subset_df[['Monthly_Spend', 'Discount_Applied']])

DataFrame after safe .loc assignment:
            Support_Tickets  High_Support_Flag
Account_ID                                    
ACC-100                   3                1.0
ACC-101                   0                0.0
ACC-102                   1                0.0
ACC-103                   4                1.0
ACC-104                   0                0.0
ACC-105                   2                0.0
ACC-106                   1                0.0
ACC-107                   5                1.0

Isolated subset DataFrame:
             Monthly_Spend  Discount_Applied
Account_ID                                 
ACC-101             120.5            108.45
ACC-104             150.2            135.18


---
### 5. Creating Derived Features

Feature engineering often involves creating normalized or interaction terms:

In [6]:
# Spend per support ticket interaction feature
df['Spend_Per_Ticket'] = df['Monthly_Spend'] / (df['Support_Tickets'] + 1)

# Is Senior Citizen binary indicator
df['Is_Senior'] = (df['Age'] >= 60).astype(int)

print("Engineered Features Added:")
print(df[['Age', 'Is_Senior', 'Monthly_Spend', 'Support_Tickets', 'Spend_Per_Ticket']])

Engineered Features Added:
            Age  Is_Senior  Monthly_Spend  Support_Tickets  Spend_Per_Ticket
Account_ID                                                                  
ACC-100      24          0           45.2                3         11.300000
ACC-101      52          0          120.5                0        120.500000
ACC-102      36          0           89.0                1         44.500000
ACC-103      41          0           34.0                4          6.800000
ACC-104      67          1          150.2                0        150.200000
ACC-105      29          0           65.4                2         21.800000
ACC-106      38          0          112.0                1         56.000000
ACC-107      45          0           78.5                5         13.083333


---
### 6. Advanced Complex Usage: Method Chaining, Dynamic Queries, and Multi-Branch Vectorization

In production data pipelines, writing intermediate variables at every step litters memory and obscures the logical data flow.

Advanced Pandas practitioners leverage:
1. **Method Chaining** via `.assign()` and `.pipe()`: Composes declarative transformations into a single readable, functional sequence.
2. **Dynamic `.query()` with `@` variables**: Allows dynamic runtime parameters from external scopes.
3. **Multi-Branch Vectorization (`np.select`)**: Evaluates multiple non-overlapping conditions simultaneously in $O(N)$ vectorized NumPy speed rather than slow `.apply()` loops.

In [7]:
# Advanced Method Chaining Pipeline with .pipe()
def filter_active_cohort(data: pd.DataFrame, min_spend: float) -> pd.DataFrame:
    # Filter records above threshold spend.
    return data[data['Monthly_Spend'] >= min_spend].copy()

def categorize_churn_risk(data: pd.DataFrame) -> pd.DataFrame:
    # Vectorized multi-tier risk classification using np.select.
    conditions = [
        (data['Support_Tickets'] >= 3) & (data['Contract'] == 'Month-to-month'),
        (data['Monthly_Spend'] > 100) & (data['Support_Tickets'] >= 1),
        (data['Contract'] == 'Two year')
    ]
    choices = ['CRITICAL_RISK', 'ELEVATED_RISK', 'LOW_RISK']
    
    # Fast vectorized branch selection with default fallback
    return data.assign(
        Risk_Category=np.select(conditions, choices, default='MODERATE_RISK')
    )

# Execute full declarative pipeline
threshold = 40.0  # Dynamic external parameter
processed_df = (
    df
    .pipe(filter_active_cohort, min_spend=threshold)
    .assign(
        Log_Monthly_Spend=lambda d: np.log1p(d['Monthly_Spend']),
        Spend_Zscore=lambda d: (d['Monthly_Spend'] - d['Monthly_Spend'].mean()) / d['Monthly_Spend'].std()
    )
    .pipe(categorize_churn_risk)
    .sort_values(by='Monthly_Spend', ascending=False)
)

print("Advanced Method-Chained Output DataFrame:")
print(processed_df[['Monthly_Spend', 'Spend_Zscore', 'Risk_Category']])

# Dynamic query using @ external parameter reference
target_risk = 'CRITICAL_RISK'
flagged = processed_df.query("Risk_Category == @target_risk and Monthly_Spend > @threshold")
print(f"\nFiltered dynamically using query(@target_risk):")
print(flagged[['Monthly_Spend', 'Support_Tickets', 'Risk_Category']])

Advanced Method-Chained Output DataFrame:
            Monthly_Spend  Spend_Zscore  Risk_Category
Account_ID                                            
ACC-104             150.2      1.562894       LOW_RISK
ACC-101             120.5      0.731031       LOW_RISK
ACC-106             112.0      0.492956  ELEVATED_RISK
ACC-102              89.0     -0.151248  MODERATE_RISK
ACC-107              78.5     -0.445341  CRITICAL_RISK
ACC-105              65.4     -0.812257  MODERATE_RISK
ACC-100              45.2     -1.378036  CRITICAL_RISK

Filtered dynamically using query(@target_risk):
            Monthly_Spend  Support_Tickets  Risk_Category
Account_ID                                               
ACC-107              78.5                5  CRITICAL_RISK
ACC-100              45.2                3  CRITICAL_RISK


### Summary & Next Steps
In this notebook, you mastered:
- Explicit index selection using `.loc` and `.iloc`.
- Compound boolean logic and high-level query operators (`.isin`, `.between`, `.query`).
- Eliminating `SettingWithCopyWarning` via explicit coordinate assignment.
- Production method chaining pipelines with `.pipe()`, `.assign()`, dynamic querying, and multi-branch `np.select` vectorization.

**Next Notebook:** `03_data_cleaning_and_missing_values.ipynb` — Handle missing data imputation, duplicate reconciliation, and text preprocessing.